<a href="https://colab.research.google.com/github/CChinmayi18/red_black_tree_dsa_/blob/main/Red_Black_Tree_DSA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import math

RED = "RED"
BLACK = "BLACK"


class Node:
    def __init__(self, value, color=RED):
        self.value = value
        self.color = color
        self.left = None
        self.right = None
        self.parent = None


class RedBlackTree:
    def __init__(self):
        self.NIL = Node(None, BLACK)
        self.NIL.left = self.NIL
        self.NIL.right = self.NIL
        self.NIL.parent = self.NIL
        self.root = self.NIL

    def trace(self, message):
        print("[TRACE]", message)

    # Left rotation
    def left_rotate(self, x):
        y = x.right
        self.trace(f"Left rotation at {x.value}")

        x.right = y.left
        if y.left != self.NIL:
            y.left.parent = x

        y.parent = x.parent

        if x.parent == self.NIL:
            self.root = y
        elif x == x.parent.left:
            x.parent.left = y
        else:
            x.parent.right = y

        y.left = x
        x.parent = y

    # Right rotation
    def right_rotate(self, y):
        x = y.left
        self.trace(f"Right rotation at {y.value}")

        y.left = x.right
        if x.right != self.NIL:
            x.right.parent = y

        x.parent = y.parent

        if y.parent == self.NIL:
            self.root = x
        elif y == y.parent.right:
            y.parent.right = x
        else:
            y.parent.left = x

        x.right = y
        y.parent = x

    # Insertion
    def insert(self, value):
        parent = self.NIL
        current = self.root

        while current != self.NIL:
            parent = current

            if value == current.value:
                self.trace("Duplicate value. Insertion skipped.")
                return

            if value < current.value:
                current = current.left
            else:
                current = current.right

        node = Node(value)
        node.left = self.NIL
        node.right = self.NIL
        node.parent = parent

        if parent == self.NIL:
            self.root = node
        elif value < parent.value:
            parent.left = node
        else:
            parent.right = node

        self.trace(f"Inserted {value} as RED.")
        self.insert_fixup(node)

    # Restore Red-Black properties after insertion
    def insert_fixup(self, z):
        while z.parent.color == RED:
            if z.parent == z.parent.parent.left:
                uncle = z.parent.parent.right

                # Case 1: Red uncle
                if uncle.color == RED:
                    self.trace("Insertion: recoloring (Case 1).")
                    z.parent.color = BLACK
                    uncle.color = BLACK
                    z.parent.parent.color = RED
                    z = z.parent
                    z = z.parent

                else:
                    # Case 2: Left-Right triangle
                    if z == z.parent.right:
                        self.trace("Insertion: triangle (Case 2).")
                        z = z.parent
                        self.left_rotate(z)

                    # Case 3: Left-Left line
                    self.trace("Insertion: line (Case 3).")
                    z.parent.color = BLACK
                    z.parent.parent.color = RED
                    self.right_rotate(z.parent.parent)

            else:
                uncle = z.parent.parent.left

                # Case 1: Red uncle
                if uncle.color == RED:
                    self.trace("Insertion: recoloring (mirror Case 1).")
                    z.parent.color = BLACK
                    uncle.color = BLACK
                    z.parent.parent.color = RED
                    z = z.parent.parent

                else:
                    # Case 2: Right-Left triangle
                    if z == z.parent.left:
                        self.trace("Insertion: triangle (mirror Case 2).")
                        z = z.parent
                        self.right_rotate(z)

                    # Case 3: Right-Right line
                    self.trace("Insertion: line (mirror Case 3).")
                    z.parent.color = BLACK
                    z.parent.parent.color = RED
                    self.left_rotate(z.parent.parent)

        self.root.color = BLACK

    # Find minimum node
    def minimum(self, node):
        while node.left != self.NIL:
            node = node.left
        return node

    # Replace one subtree with another
    def transplant(self, u, v):
        if u.parent == self.NIL:
            self.root = v
        elif u == u.parent.left:
            u.parent.left = v
        else:
            u.parent.right = v

        v.parent = u.parent

    # Deletion
    def delete(self, value):
        z = self.root

        while z != self.NIL and z.value != value:
            if value < z.value:
                z = z.left
            else:
                z = z.right

        if z == self.NIL:
            print("Value not found.")
            return

        if z.left == self.NIL and z.right == self.NIL:
            self.trace("Deletion Case 1: leaf node.")
        elif z.left == self.NIL or z.right == self.NIL:
            self.trace("Deletion Case 2: node with one child.")
        else:
            self.trace("Deletion Case 3: node with two children.")

        y = z
        original_color = y.color

        if z.left == self.NIL:
            x = z.right
            self.transplant(z, z.right)

        elif z.right == self.NIL:
            x = z.left
            self.transplant(z, z.left)

        else:
            y = self.minimum(z.right)
            original_color = y.color
            x = y.right

            if y.parent == z:
                x.parent = y
            else:
                self.transplant(y, y.right)
                y.right = z.right
                y.right.parent = y

            self.transplant(z, y)
            y.left = z.left
            y.left.parent = y
            y.color = z.color

        if original_color == BLACK:
            self.trace("Deletion fix-up required.")
            self.delete_fixup(x)

        self.NIL.parent = self.NIL

        if self.root != self.NIL:
            self.root.color = BLACK

        print(f"Deleted {value} successfully.")

    # Restore Red-Black properties after deletion
    def delete_fixup(self, x):
        while x != self.root and x.color == BLACK:

            if x == x.parent.left:
                w = x.parent.right

                # Case 1: Red sibling
                if w.color == RED:
                    self.trace("Delete fix-up Case 1.")
                    w.color = BLACK
                    x.parent.color = RED
                    self.left_rotate(x.parent)
                    w = x.parent.right

                # Case 2: Both sibling children are black
                if (w.left.color == BLACK and
                        w.right.color == BLACK):
                    self.trace("Delete fix-up Case 2.")
                    if w != self.NIL:
                        w.color = RED
                    x = x.parent

                else:
                    # Case 3: Near child red, far child black
                    if w.right.color == BLACK:
                        self.trace("Delete fix-up Case 3.")
                        w.left.color = BLACK
                        w.color = RED
                        self.right_rotate(w)
                        w = x.parent.right

                    # Case 4: Far child red
                    self.trace("Delete fix-up Case 4.")
                    w.color = x.parent.color
                    x.parent.color = BLACK
                    w.right.color = BLACK
                    self.left_rotate(x.parent)
                    x = self.root

            else:
                w = x.parent.left

                # Case 1: Red sibling
                if w.color == RED:
                    self.trace("Delete fix-up Case 1 (mirror).")
                    w.color = BLACK
                    x.parent.color = RED
                    self.right_rotate(x.parent)
                    w = x.parent.left

                # Case 2: Both sibling children are black
                if (w.right.color == BLACK and
                        w.left.color == BLACK):
                    self.trace("Delete fix-up Case 2 (mirror).")
                    if w != self.NIL:
                        w.color = RED
                    x = x.parent

                else:
                    # Case 3: Near child red, far child black
                    if w.left.color == BLACK:
                        self.trace("Delete fix-up Case 3 (mirror).")
                        w.right.color = BLACK
                        w.color = RED
                        self.left_rotate(w)
                        w = x.parent.left

                    # Case 4: Far child red
                    self.trace("Delete fix-up Case 4 (mirror).")
                    w.color = x.parent.color
                    x.parent.color = BLACK
                    w.left.color = BLACK
                    self.right_rotate(x.parent)
                    x = self.root

        x.color = BLACK

    # Calculate height in number of nodes
    def height(self, node):
        if node == self.NIL:
            return 0

        return 1 + max(
            self.height(node.left),
            self.height(node.right)
        )

    # Count nodes
    def count_nodes(self, node):
        if node == self.NIL:
            return 0

        return (
            1
            + self.count_nodes(node.left)
            + self.count_nodes(node.right)
        )

    # Display tree structure and height analysis
    def display(self):
        if self.root == self.NIL:
            print("Tree is empty.")
            return

        def show(node, prefix="", is_left=True):
            if node == self.NIL:
                return

            show(node.right, prefix + "    ", False)

            branch = "└── " if is_left else "┌── "
            print(prefix + branch +
                  f"{node.value} [{node.color}]")

            show(node.left, prefix + "    ", True)

        print("\nTree structure:")
        show(self.root)

        n = self.count_nodes(self.root)
        h = self.height(self.root)
        bound = 2 * math.log2(n + 1)

        print("\nRoot:", self.root.value)
        print("Root Color:", self.root.color)
        print("Number of Nodes:", n)
        print("Tree Height:", h)
        print("Theoretical Height Bound:", round(bound, 2))
        print("Height Balance:",
              "Satisfied" if h <= bound else "Not Satisfied")


# Main menu
def main():
    tree = None

    while True:
        print("\n===== RED-BLACK TREE =====")
        print("1. Create tree")
        print("2. Insert node")
        print("3. Delete node")
        print("4. Display tree")
        print("5. Exit")

        choice = input("Enter choice: ").strip()

        if choice == "1":
            tree = RedBlackTree()
            print("New empty Red-Black Tree created.")

            values = input(
                "Enter initial values separated by spaces: "
            ).split()

            try:
                for value in values:
                    tree.insert(int(value))

                print("Tree creation completed.")
                tree.display()

            except ValueError:
                print("Invalid input. Enter integers only.")

        elif choice == "2":
            if tree is None:
                print("Create a tree first.")
                continue

            try:
                value = int(input("Enter value to insert: "))
                tree.insert(value)
                tree.display()
            except ValueError:
                print("Enter a valid integer.")

        elif choice == "3":
            if tree is None:
                print("Create a tree first.")
                continue

            try:
                value = int(input("Enter value to delete: "))
                tree.delete(value)
                tree.display()
            except ValueError:
                print("Enter a valid integer.")

        elif choice == "4":
            if tree is None:
                print("Create a tree first.")
            else:
                tree.display()

        elif choice == "5":
            print("Exiting program.")
            break

        else:
            print("Invalid choice. Enter 1-5.")


if __name__ == "__main__":
    main()



===== RED-BLACK TREE =====
1. Create tree
2. Insert node
3. Delete node
4. Display tree
5. Exit
Enter choice: 1
New empty Red-Black Tree created.
Enter initial values separated by spaces: 40 20 60 10 30 50 70
[TRACE] Inserted 40 as RED.
[TRACE] Inserted 20 as RED.
[TRACE] Inserted 60 as RED.
[TRACE] Inserted 10 as RED.
[TRACE] Insertion: recoloring (Case 1).
[TRACE] Inserted 30 as RED.
[TRACE] Inserted 50 as RED.
[TRACE] Inserted 70 as RED.
Tree creation completed.

Tree structure:
        ┌── 70 [RED]
    ┌── 60 [BLACK]
        └── 50 [RED]
└── 40 [BLACK]
        ┌── 30 [RED]
    └── 20 [BLACK]
        └── 10 [RED]

Root: 40
Root Color: BLACK
Number of Nodes: 7
Tree Height: 3
Theoretical Height Bound: 6.0
Height Balance: Satisfied

===== RED-BLACK TREE =====
1. Create tree
2. Insert node
3. Delete node
4. Display tree
5. Exit
Enter choice: 2
Enter value to insert: 25
[TRACE] Inserted 25 as RED.
[TRACE] Insertion: recoloring (mirror Case 1).

Tree structure:
        ┌── 70 [RED]
    ┌